# Real integration test: `get_base_model(partial=True)`

Downloads DeepSeek-V2-Lite from HuggingFace, builds a partial model through the real `get_base_model` code path with a real `ExpertManager` loaded from `expert_groups/exp_dummy`, and verifies that backbone + owned-expert tensors match the pretrained checkpoint instead of the random `_init_weights` default.

CPU-only — every weight load and tensor comparison runs in host RAM. VRAM is never touched: the script sets `config.model.device = "cpu"`, so `get_base_model` keeps the partial model on CPU and the pretrained state dict is also loaded on CPU (production miners with a real GPU would land the partial model in VRAM instead, leaving the CPU state dict to drain into VRAM tensor-by-tensor as it streams).

No mocks. Heavy — peak ~40 GB RAM during the streaming load (partial model + the still-being-consumed pretrained state dict), and another ~40 GB peak during the verification compare. First run pulls ~16 GB from HuggingFace; subsequent runs reuse the HF cache.

Run this notebook from the repo root so the `connito` package is importable.

In [1]:
from __future__ import annotations

from pathlib import Path

import torch

from connito.shared.config import MinerConfig
from connito.shared.expert_manager import ExpertManager
from connito.shared.modeling.custom_deepseek_v2_lite import CustomDeekSeekMoE
from connito.shared.modeling.mycelia import (
    get_base_model,
    get_base_tokenizer,
    load_pretrained_state_dict,
)

/home/isabella/crucible/ConnitoAI/.main-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "expert_groups" / "exp_math").is_dir():
    REPO_ROOT = REPO_ROOT.parent

EXP_DUMMY_DIR = REPO_ROOT / "expert_groups" / "exp_math"
MODEL_PATH = "deepseek-ai/DeepSeek-V2-Lite"
GROUP_ID = 0

assert EXP_DUMMY_DIR.is_dir(), f"expert_groups/exp_math not found at {EXP_DUMMY_DIR}"
print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"EXP_DUMMY_DIR = {EXP_DUMMY_DIR}")

REPO_ROOT     = /home/isabella/crucible/ConnitoAI
EXP_DUMMY_DIR = /home/isabella/crucible/ConnitoAI/expert_groups/exp_math


## Helpers

`_assert` prints a PASS/FAIL line and raises on failure. The compare helpers walk the partial model's state dict against the pretrained state dict for backbone keys and against the per-expert assignment map for owned experts.

In [3]:
def _assert(condition: bool, message: str) -> None:
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {message}")
    if not condition:
        raise AssertionError(message)


def _compare_backbone(partial_state: dict, full_state: dict) -> None:
    """Walk every key shared between the partial model and the pretrained HF
    state dict; for shape-matched keys, assert the partial model's tensor
    equals the pretrained tensor."""
    shared = [k for k in partial_state if k in full_state]
    matched = []
    mismatched: list[str] = []
    skipped_shape: list[str] = []

    for key in shared:
        p = partial_state[key]
        f = full_state[key]
        if tuple(p.shape) != tuple(f.shape):
            skipped_shape.append(key)
            continue
        if torch.equal(p.detach().cpu().to(f.dtype), f.detach().cpu()):
            matched.append(key)
        else:
            mismatched.append(key)

    print(f"  shape-matched backbone keys: {len(matched)} / {len(shared) - len(skipped_shape)}")
    if matched:
        print(f"  first 5 matched keys: {matched[:50]}")
    if mismatched:
        print(f"  first 5 mismatched keys: {mismatched[:5]}")
    if skipped_shape:
        print(f"  shape-mismatched keys (expected — sliced experts): {len(skipped_shape)}")

    _assert(
        len(matched) > 0,
        "at least one backbone tensor matches the pretrained checkpoint",
    )
    _assert(
        not mismatched,
        f"every shape-matched key equals the pretrained value (got {len(mismatched)} mismatches)",
    )


def _compare_owned_experts(
    partial_model: CustomDeekSeekMoE,
    full_state: dict,
    expert_group_assignment: dict,
    target_group: int,
) -> None:
    """For each owned (my_idx, org_idx) pair, verify the partial model's
    expert tensor equals the pretrained expert at org_idx."""
    partial_state = partial_model.state_dict()
    assignments = expert_group_assignment.get(target_group, {})

    checked = 0
    mismatched: list[str] = []
    missing_full: list[str] = []

    for layer_idx, layer_pairs in assignments.items():
        if not layer_pairs:
            continue
        for my_idx, org_idx in layer_pairs:
            print(f"  Checking owned expert {my_idx} (pretrained index {org_idx})")
            for proj in ("gate_proj", "up_proj", "down_proj"):
                key = f"model.layers.{layer_idx}.mlp.experts.{my_idx}.{proj}.weight"
                if key not in partial_state:
                    print(f"  Skipping {key} (not in partial state)")
                    continue
                full_key_per_expert = (
                    f"model.layers.{layer_idx}.mlp.experts.{org_idx}.{proj}.weight"
                )
                if full_key_per_expert not in full_state:
                    missing_full.append(full_key_per_expert)
                    print(f"  Skipping {full_key_per_expert} (not in pretrained state)")
                    continue
                p = partial_state[key].detach().cpu()
                f = full_state[full_key_per_expert].detach().cpu()
                if tuple(p.shape) != tuple(f.shape):
                    continue
                if torch.equal(p.to(f.dtype), f):
                    checked += 1
                else:
                    mismatched.append(key)

    print(f"  owned-expert tensors verified: {checked} {len(mismatched)} mismatches")
    
    if mismatched:
        print(f"  first 5 owned-expert mismatches: {mismatched[:5]}")
    
    if missing_full:
        print(f"  pretrained-side keys not found (skipped): {len(missing_full)}")

    _assert(
        checked > 0,
        "at least one owned-expert tensor matches the pretrained slice",
    )
    _assert(
        not mismatched,
        f"every checked owned-expert equals the pretrained slice (got {len(mismatched)} mismatches)",
    )


def _check_not_random(partial_state: dict) -> None:
    """Sanity: HuggingFace `_init_weights` for DeepSeek uses
    `normal_(std=config.initializer_range)` (~0.02). The pretrained embedding
    table has a noticeably different distribution. If the fix regressed and
    weights came back random, every backbone std would sit very close to 0.02."""
    key = "model.embed_tokens.weight"
    _assert(key in partial_state, f"partial state dict contains `{key}`")
    std = float(partial_state[key].detach().float().std().item())
    print(f"  embed_tokens.weight std = {std:.4f}")
    _assert(
        abs(std - 0.02) > 0.005,
        "embed_tokens.weight std differs from the random `_init_weights` default of ~0.02",
    )


def _run_forward_pass(
    partial_model: CustomDeekSeekMoE,
    config: MinerConfig,
) -> None:
    """Run a single CPU forward pass through the partial model.

    Structural smoke test: the partial model only owns the experts for one
    group, and `myceliaSparseMoeBlock` silently skips tokens routed to
    unowned experts (`local_expert_idx == -1`), so the logits are only
    partially informed. We assert the model is runnable end-to-end and
    produces a finite logits tensor of the expected `(batch, seq, vocab)`
    shape — not that the output is semantically meaningful."""
    tokenizer = get_base_tokenizer(config)
    prompt = "Hello, world!"
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"]
    print(f"  prompt        = {prompt!r}")
    print(f"  input_ids     = shape {tuple(input_ids.shape)}, dtype {input_ids.dtype}")

    partial_model.eval()
    with torch.no_grad():
        outputs = partial_model(input_ids=input_ids)

    logits = outputs.logits
    print(f"  logits        = shape {tuple(logits.shape)}, dtype {logits.dtype}")

    expected_shape = (
        input_ids.shape[0],
        input_ids.shape[1],
        partial_model.config.vocab_size,
    )
    _assert(
        tuple(logits.shape) == expected_shape,
        f"logits shape is (batch, seq, vocab) = {expected_shape}",
    )
    _assert(
        bool(torch.isfinite(logits).all().item()),
        "all logits are finite (no NaN / Inf)",
    )

## [1/7] Build `MinerConfig` + `ExpertManager` (real, from disk fixture)

`model.device = "cpu"` keeps any downstream `.to(device=...)` calls off CUDA. `get_base_model` itself never moves the model anywhere, but setting this explicitly makes the CPU-only intent clear.

In [6]:
group_id = GROUP_ID  # change here to test a different group from exp_dummy

print("=" * 72)
print("Real integration test: get_base_model(partial=True)")
print(f"  model_path   = {MODEL_PATH}")
print(f"  group_id     = {group_id}")
print(f"  fixture      = {EXP_DUMMY_DIR}")
print( "  device       = cpu (VRAM untouched)")
print("=" * 72)

config = MinerConfig()
config.role = "miner"
config.model.model_path = MODEL_PATH
config.model.base_arch_model = MODEL_PATH
config.model.device = "cpu"
config.model.precision = "bf16-mixed"
# config.model.use_quantization = False
# config.model.use_unsloth = False
config.model.torch_compile = False
config.task.expert_group_name = "exp_math"
config.task.base_path = EXP_DUMMY_DIR.parent
config.task.path = EXP_DUMMY_DIR
config.task.load_all_expert_groups = False
config.task.exp.group_id = group_id
config.task.exp.data.sequence_length = 128

print("\n[1/7] Building MinerConfig + ExpertManager (real, from disk fixture)")
expert_manager = ExpertManager(config)
n_groups = expert_manager.num_expert_groups
print(f"  expert_group_assignment loaded ({n_groups} group(s))")
_assert(
    group_id in expert_manager.expert_group_assignment,
    f"expert_manager.expert_group_assignment contains group_id={group_id}",
)

Real integration test: get_base_model(partial=True)
  model_path   = deepseek-ai/DeepSeek-V2-Lite
  group_id     = 0
  fixture      = /home/isabella/crucible/ConnitoAI/expert_groups/exp_math
  device       = cpu (VRAM untouched)
2026-05-12 14:30:34 [info     ] Resolving wallet data from chain coldkey=template_coldkey_name hotkey=template_hotkey_name network=archive
2026-05-12 14:30:37 [warning  ] Cannot find wallet keys; check coldkey/hotkey names or pass --hotkey_name/--coldkey_name coldkey_name=template_coldkey_name error=Generic error: Failed to get hotkey: FileNotFound("Keyfile at: /home/isabella/.bittensor/wallets/template_coldkey_name/hotkeys/template_hotkey_name does not exist.") hotkey_name=template_hotkey_name

[1/7] Building MinerConfig + ExpertManager (real, from disk fixture)
2026-05-12 14:30:37 [info     ] Loading expert assignment for active task only task_path=/home/isabella/crucible/ConnitoAI/expert_groups/exp_math
  expert_group_assignment loaded (1 group(s))
  [PASS] 

## [2/7] Call `get_base_model(partial=True)`

In [7]:
print("\n[2/7] Calling get_base_model(partial=True)")
partial_model = get_base_model(
    config=config,
    expert_manager=expert_manager,
    group_ids=[group_id],
    partial=True,
)
_assert(partial_model is not None, "get_base_model returned a model")
_assert(
    isinstance(partial_model, CustomDeekSeekMoE),
    f"model is CustomDeekSeekMoE (got {type(partial_model).__name__})",
)

partial_state = partial_model.state_dict()
print(f"  partial model state_dict keys: {len(partial_state)}")


[2/7] Calling get_base_model(partial=True)


2026-05-12 14:30:44 [info     ] Initialized MoE expert layout expert_first=4 expert_last=46 full_mode=False layer_id=1 num_local_experts=8 position=first
2026-05-12 14:30:56 [info     ] Initialized MoE expert layout expert_first=9 expert_last=60 full_mode=False layer_id=26 num_local_experts=8 position=last
2026-05-12 14:31:12 [info     ] Streamed pretrained safetensors shards into partial model loaded_counts={'full': 299, 'sliced': 0} shards=4 target_group=0
2026-05-12 14:31:12 [info     ] Streamed pretrained safetensors directly into partial model device=cpu dtype=torch.bfloat16 target_group=0
  [PASS] get_base_model returned a model
  [PASS] model is CustomDeekSeekMoE (got CustomDeekSeekMoE)
  partial model state_dict keys: 715


## [3/7] Sanity-check: weights are not at the random `_init_weights` default

In [8]:
print("\n[3/7] Sanity-check: weights are not at the random `_init_weights` default")
_check_not_random(partial_state)


[3/7] Sanity-check: weights are not at the random `_init_weights` default
  [PASS] partial state dict contains `model.embed_tokens.weight`
The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
  embed_tokens.weight std = 0.1315
  [PASS] embed_tokens.weight std differs from the random `_init_weights` default of ~0.02


## [4/7] Compare partial model tensors against the HF pretrained checkpoint

First run pulls ~16 GB from HuggingFace; subsequent runs reuse the HF cache. We load the state dict in bf16 to match the upstream safetensors storage exactly (no load-time cast). Peak RAM during this cell is ~34 GB.

In [9]:
print("\n[4/7] Comparing partial model tensors against the HF pretrained checkpoint")
print("  Loading pretrained state dict in bf16 on CPU (first run pulls ~16 GB)...")
full_state = load_pretrained_state_dict(MODEL_PATH, dtype=torch.bfloat16)
print(f"  pretrained state dict keys: {len(full_state)}")

`torch_dtype` is deprecated! Use `dtype` instead!



[4/7] Comparing partial model tensors against the HF pretrained checkpoint
  Loading pretrained state dict in bf16 on CPU (first run pulls ~16 GB)...


Loading weights: 100%|██████████| 351/351 [00:04<00:00, 70.70it/s]


  pretrained state dict keys: 351


In [10]:
for k in partial_state.keys():
    if "expert" in k:
        print(f"expert key {k}")
        break

for k in full_state.keys():
    if "expert" in k:
        print(f"expert key {k}")
        break

expert key model.layers.1.mlp.experts.4.gate_up_proj
expert key model.layers.1.mlp.experts.gate_up_proj


In [11]:
print("\n  --- Backbone ---")
_compare_backbone(partial_state, full_state)


  --- Backbone ---
  shape-matched backbone keys: 299 / 299
  first 5 matched keys: ['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.kv_a_proj_with_mqa.weight', 'model.layers.0.self_attn.kv_a_layernorm.weight', 'model.layers.0.self_attn.kv_b_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.kv_a_proj_with_mqa.weight', 'model.layers.1.self_attn.kv_a_layernorm.weight', 'model.layers.1.self_attn.kv_b_proj.weight', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.mlp.gate.weight', 'model.layers.1.mlp.shared_experts.gate_proj.weight', 'model.layers.1.mlp.shared_experts.up_proj.weight', 'model.layers.1.mlp.shared_experts.down_proj.weight', 'model.layers.1.inp

## [5/7] Forward pass smoke test (CPU)

Tokenize a short prompt, run a single forward pass through the partial model with `torch.no_grad()`, and verify the logits tensor has shape `(batch, seq, vocab)` and contains only finite values. Most tokens will route to experts the partial model doesn't own — `myceliaSparseMoeBlock` silently skips those contributions, so the logits aren't semantically meaningful. This is a structural check that the partial model is runnable end-to-end.

In [12]:
print("\n[5/7] Forward pass smoke test (CPU)")
_run_forward_pass(partial_model, config)


[5/7] Forward pass smoke test (CPU)


  prompt        = 'Hello, world!'
  input_ids     = shape (1, 4), dtype torch.int64
  logits        = shape (1, 4, 102400), dtype torch.bfloat16
  [PASS] logits shape is (batch, seq, vocab) = (1, 4, 102400)
  [PASS] all logits are finite (no NaN / Inf)


## [6/7] Brief eval: partial vs full (prompt-based)

Compute mean next-token cross-entropy loss for a handful of short prompts, first with the **partial** model already loaded, then with the **full** HF model. The partial model only owns experts for one group, so `myceliaSparseMoeBlock` silently zeros out contributions from unowned experts — the partial loss is expected to be substantially higher than the full-model loss. The comparison is a qualitative sanity check that the streamed weights reach the right places (full model loss should land in a normal range for DeepSeek-V2-Lite on English text).

Run sequentially: keep the partial model resident, compute its losses, then load the full HF model (~30 GB bf16 on CPU) and compute the same losses. Peak host RAM during this section is roughly the partial model + the full model + any leftover safetensors mmap.

In [13]:
EVAL_PROMPTS = [
    "The quick brown fox jumps over the lazy dog near the riverbank.",
    "Machine learning models can be fine-tuned for downstream tasks like classification and summarization.",
    "In a quiet village by the mountains, an old librarian guarded a collection of rare maps.",
    "Photosynthesis converts sunlight into chemical energy stored as glucose inside plant cells.",
]


def _compute_eval_loss(model, tokenizer, prompts, label):
    """Mean next-token cross-entropy across `prompts`. Per-prompt loss is
    printed so a divergent prompt is obvious. Returns the mean loss and
    the per-prompt list."""
    model.eval()
    per_prompt = []
    with torch.no_grad():
        for prompt in prompts:
            inputs = tokenizer(prompt, return_tensors="pt")
            input_ids = inputs["input_ids"]
            outputs = model(input_ids=input_ids, labels=input_ids)
            loss = float(outputs.loss.item())
            per_prompt.append(loss)
            print(f"  [{label}] loss={loss:7.4f}  tokens={input_ids.shape[1]:>3}  {prompt!r}")
    mean_loss = sum(per_prompt) / len(per_prompt)
    print(f"  [{label}] mean loss = {mean_loss:.4f}")
    return mean_loss, per_prompt


print("\n[6/7] Brief eval: partial model")
tokenizer = get_base_tokenizer(config)
partial_mean, partial_losses = _compute_eval_loss(
    model=partial_model,
    tokenizer=tokenizer,
    prompts=EVAL_PROMPTS,
    label="partial",
)


[6/7] Brief eval: partial model


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  [partial] loss=10.5581  tokens= 14  'The quick brown fox jumps over the lazy dog near the riverbank.'
  [partial] loss=10.6099  tokens= 17  'Machine learning models can be fine-tuned for downstream tasks like classification and summarization.'
  [partial] loss=12.9006  tokens= 18  'In a quiet village by the mountains, an old librarian guarded a collection of rare maps.'
  [partial] loss=10.7428  tokens= 14  'Photosynthesis converts sunlight into chemical energy stored as glucose inside plant cells.'
  [partial] mean loss = 11.2029


In [14]:
from transformers import AutoModelForCausalLM

print("\n  Loading full HF model in bf16 on CPU (reuses HF cache from the earlier load)...")
full_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    # trust_remote_code=True,
)
full_model.eval()
print(f"  full model class: {type(full_model).__name__}")
print(f"  full model dtype: {next(full_model.parameters()).dtype}")


  Loading full HF model in bf16 on CPU (reuses HF cache from the earlier load)...


Loading weights: 100%|██████████| 351/351 [00:04<00:00, 71.75it/s]

  full model class: DeepseekV2ForCausalLM
  full model dtype: torch.bfloat16


In [15]:
print("\n[6/7] Brief eval: full model")
full_mean, full_losses = _compute_eval_loss(
    model=full_model,
    tokenizer=tokenizer,
    prompts=EVAL_PROMPTS,
    label="full",
)

print("\n  --- Side-by-side ---")
print(f"  {'prompt':<70}  {'partial':>9}  {'full':>9}  {'delta':>9}")
for prompt, p_loss, f_loss in zip(EVAL_PROMPTS, partial_losses, full_losses):
    short = prompt if len(prompt) <= 70 else prompt[:67] + "..."
    print(f"  {short:<70}  {p_loss:>9.4f}  {f_loss:>9.4f}  {p_loss - f_loss:>+9.4f}")
print(f"  {'mean':<70}  {partial_mean:>9.4f}  {full_mean:>9.4f}  {partial_mean - full_mean:>+9.4f}")


[6/7] Brief eval: full model


  [full] loss= 2.7215  tokens= 14  'The quick brown fox jumps over the lazy dog near the riverbank.'
  [full] loss= 2.4723  tokens= 17  'Machine learning models can be fine-tuned for downstream tasks like classification and summarization.'
  [full] loss= 3.7257  tokens= 18  'In a quiet village by the mountains, an old librarian guarded a collection of rare maps.'
  [full] loss= 2.4311  tokens= 14  'Photosynthesis converts sunlight into chemical energy stored as glucose inside plant cells.'
  [full] mean loss = 2.8376

  --- Side-by-side ---
  prompt                                                                    partial       full      delta
  The quick brown fox jumps over the lazy dog near the riverbank.           10.5581     2.7215    +7.8366
  Machine learning models can be fine-tuned for downstream tasks like...    10.6099     2.4723    +8.1376
  In a quiet village by the mountains, an old librarian guarded a col...    12.9006     3.7257    +9.1749
  Photosynthesis converts sun

## [7/7] Validator-style eval: `get_dataloader` + `evaluate_model`

Runs the same eval pipeline `validator/run.py` uses in production:

1. `get_dataloader(config, tokenizer, seed, rank=0, world_size=1, train=False)` builds the streaming HF eval loader.
2. We materialize the first few batches into a list so both models see the same data (mirrors `evaluator.py`'s `cached_batches` pattern).
3. `evaluate_model(step, model, eval_dataloader, device, max_eval_batches, rank)` returns `{"val_loss": float}` — the same number a validator scores miners against.

This is slower per batch than the prompt-based eval above (full-length sequences from the streaming dataset, both models on CPU), so we cap at a handful of batches. Keep `N_BATCHES` small unless you have time to wait.

In [17]:
from connito.shared.dataloader import get_dataloader
from connito.shared.evaluate import evaluate_model

EVAL_SEED = 42
N_BATCHES = 10  # brief — CPU forward on DeepSeek-V2-Lite is slow

device = torch.device("cpu")

print("\n[7/7] Building eval dataloader (validator-style)")
eval_loader = get_dataloader(
    config=config,
    tokenizer=tokenizer,
    seed=EVAL_SEED,
    rank=0,
    world_size=1,
    train=False,
)

# Materialize a small fixed batch list once so partial- and full-model
# eval runs see identical inputs. Mirrors `validator/evaluator.py`'s
# `cached_batches` pattern.
cached_batches = []
for i, batch in enumerate(eval_loader):
    if i >= N_BATCHES:
        break
    cached_batches.append(batch)

assert cached_batches, "eval dataloader produced no batches"
input_shape = tuple(cached_batches[0]["input_ids"].shape)
print(f"  cached {len(cached_batches)} batches, input_ids shape = {input_shape}")


[7/7] Building eval dataloader (validator-style)


2026-05-12 14:40:21 [warning  ] Split 'validation' not found for nvidia/Nemotron-CC-Math-v1. Falling back to 'train' split.
  cached 10 batches, input_ids shape = (1, 128)


In [18]:
print("\n[7/7] Running evaluate_model on partial model")
partial_metrics = evaluate_model(
    step=0,
    model=partial_model,
    eval_dataloader=cached_batches,
    device=device,
    max_eval_batches=N_BATCHES,
    rank=0,
)
partial_val_loss = float(partial_metrics.get("val_loss", float("nan")))
print(f"  partial val_loss = {partial_val_loss:.4f}")


[7/7] Running evaluate_model on partial model


  partial val_loss = 8.2188


In [19]:
print("\n[7/7] Running evaluate_model on full model")
full_metrics = evaluate_model(
    step=0,
    model=full_model,
    eval_dataloader=cached_batches,
    device=device,
    max_eval_batches=N_BATCHES,
    rank=0,
)
full_val_loss = float(full_metrics.get("val_loss", float("nan")))
print(f"  full val_loss    = {full_val_loss:.4f}")

print("\n  --- Validator-style val_loss comparison ---")
print(f"  partial = {partial_val_loss:.4f}")
print(f"  full    = {full_val_loss:.4f}")
print(f"  delta   = {partial_val_loss - full_val_loss:+.4f}")

print("\n" + "=" * 72)
print("ALL CHECKS PASSED")
print("=" * 72)


[7/7] Running evaluate_model on full model


  full val_loss    = 2.3916

  --- Validator-style val_loss comparison ---
  partial = 8.2188
  full    = 2.3916
  delta   = +5.8271

ALL CHECKS PASSED


In [20]:
import copy 
eval_model = copy.deepcopy(partial_model)

In [21]:
  from huggingface_hub import HfApi, hf_hub_download                                                                                                                                                               
from connito.shared.helper import MINER_CHECKPOINT_SUFFIXES, load_state_dict_from_path                                                                                                                           
                                                                                                                                                                                                                   
  MINER_REPO = "savethisok/co90"                                                                                                                                                                                   
  MINER_REVISION = "main"  # branch/tag/commit hash; "main" pulls the latest                                                                                                                                       
                                                                                                                                                                                                                   
  # 1) Discover which checkpoint file the miner published.                                                                                                                                                         
  #    Convention: a single .safetensors (preferred) or .pt file in the repo.                                                                                                                                      
  api = HfApi()                                                                                                                                                                                                    
  repo_files = api.list_repo_files(MINER_REPO, revision=MINER_REVISION)
  ckpt_files = [f for f in repo_files if f.endswith(MINER_CHECKPOINT_SUFFIXES)]
  assert ckpt_files, (                                                                                    
      f"No {MINER_CHECKPOINT_SUFFIXES} files found in {MINER_REPO}@{MINER_REVISION}. "
      f"Repo files: {repo_files}"                                                                         
  )                                                  
  ckpt_files.sort(key=lambda f: 0 if f.endswith(".safetensors") else 1)  # prefer safetensors
  ckpt_name = ckpt_files[0]                                                                               
  print(f"  selected miner file: {ckpt_name}")

  # 2) Download the file into the local HF cache.                                                         
  ckpt_path = hf_hub_download(                                                                            
      repo_id=MINER_REPO,                                                                                 
      revision=MINER_REVISION,                       
      filename=ckpt_name,
  )                                                                                                       
  print(f"  downloaded to:       {ckpt_path}")

  # 3) Load the miner state dict (.safetensors via safetensors.load_file,                                 
  #    .pt via torch.load with weights_only=True — see shared/helper.py).
  miner_sd = load_state_dict_from_path(ckpt_path)                                                         
  print(f"  miner state_dict keys: {len(miner_sd)}")

  selected miner file: model_expgroup_0.safetensors
  downloaded to:       /home/isabella/.cache/huggingface/hub/models--savethisok--co90/snapshots/257fa6f5de5780a749fd527e23b4ba47f7d42655/model_expgroup_0.safetensors
  miner state_dict keys: 416


In [22]:

  # 4) Overlay onto the partial model. strict=False allows the miner's
  #    submission to cover only its assigned group's experts (and optionally
  #    backbone) — the rest of the partial model keeps the pretrained weights                             
  #    loaded by get_base_model.


  base_keys = set(eval_model.state_dict().keys())                                                      
  ckpt_keys = set(miner_sd.keys())                   

  expert_in_full = {k for k in full_model.state_dict().keys() if "expert" in k and ("share" not in k)}
  expert_in_base  = {k for k in base_keys if "expert" in k and ("share" not in k)}  # some shared expert keys are in the base model, so exclude those
  expert_in_miner = {k for k in ckpt_keys if "expert" in k and ("share" not in k)}

  common_experts = expert_in_base & expert_in_miner
  expert_not_in_base = expert_in_miner - expert_in_base

  print(f"  base keys:              {len(base_keys)}")
  print(f"  miner keys:             {len(ckpt_keys)}")                                                    
  print(f"  common keys:            {len(common_experts)}")
  print(f"  expert in full:          {len(expert_in_full)}")
  print(f"  expert in eval:          {len(expert_in_base)}")
print(f"  expert in miner:         {len(expert_in_miner)}")
print(f"  common experts:         {len(common_experts)}")
  print(f"  expert_not_in_base:    { len(sorted(expert_not_in_base))}")
  print(f"========")                
  if expert_not_in_base:                                                                                  
      print(f"  expert keys in miner but not in partial model (first 5): "
            f"{sorted(expert_not_in_base)[:5]}")                                                          

  incompatible = eval_model.load_state_dict(miner_sd, strict=False)                                    
  # print(f"  missing keys (first 5):    {list(incompatible.missing_keys)[:5]}")
  print(f"  unexpected keys (first 5): {list(incompatible.unexpected_keys)[:5]}")


  base keys:              715
  miner keys:             416
  common keys:            416
  expert in full:          52
  expert in eval:          416
  expert in miner:         416
  common experts:         416
  expert_not_in_base:    0


  unexpected keys (first 5): []


In [23]:
expert_keys = []
for k in eval_model.state_dict().keys():
    if "expert" in k:
        expert_keys.append(k)
print(f"expert keys: {len(expert_keys)}")

expert keys: 494


In [27]:
print(list(expert_in_base)[:10])

['model.layers.7.mlp.experts.54.down_proj', 'model.layers.13.mlp.experts.45.gate_up_proj', 'model.layers.23.mlp.experts.35.down_proj', 'model.layers.19.mlp.experts.12.down_proj', 'model.layers.18.mlp.experts.19.gate_up_proj', 'model.layers.14.mlp.experts.48.down_proj', 'model.layers.20.mlp.experts.18.gate_up_proj', 'model.layers.25.mlp.experts.48.gate_up_proj', 'model.layers.23.mlp.experts.47.down_proj', 'model.layers.20.mlp.experts.13.down_proj']


In [28]:
print(list(expert_in_miner)[:10])

['model.layers.7.mlp.experts.54.down_proj', 'model.layers.13.mlp.experts.45.gate_up_proj', 'model.layers.23.mlp.experts.35.down_proj', 'model.layers.19.mlp.experts.12.down_proj', 'model.layers.18.mlp.experts.19.gate_up_proj', 'model.layers.14.mlp.experts.48.down_proj', 'model.layers.20.mlp.experts.18.gate_up_proj', 'model.layers.25.mlp.experts.48.gate_up_proj', 'model.layers.23.mlp.experts.47.down_proj', 'model.layers.20.mlp.experts.13.down_proj']


In [25]:
import re                                                                                                                                                                                                        
from collections import defaultdict                                                                                                                                                                              
                                                                                                                                                                                                                   
EXPERT_KEY_RE = re.compile(r"^model\.layers\.(\d+)\.mlp\.experts(?:\.(\d+))?\.")                                                                                                                                 
                                                                                                                                                                                                                
                                                                                                                                                                                                                
def experts_per_layer(state_dict):                                                                                                                                                                               
    """{layer_idx -> set(expert_idx)} from a state dict.                                                                                                                                                         
                                                                                                                                                                                                                
    Per-expert keys (`experts.{idx}.gate_proj.weight`) contribute their                                                                                                                                          
    explicit index. Stacked keys (`experts.gate_up_proj`, no index in the                                                                                                                                        
    name) contribute 0..leading_dim-1 — for DeepSeek-V2-Lite that's 0..63.                                                                                                                                       
    Shared-expert keys are excluded.                                                                                                                                                                             
    """                                                                                                                                                                                                          
    out = defaultdict(set)                                                                                                                                                                                       
    for key, tensor in state_dict.items():                                                                                                                                                                       
        if "shared_experts" in key:                                                                                                                                                                              
            continue                                                                                                                                                                                             
        m = EXPERT_KEY_RE.match(key)                                                                                                                                                                             
        if not m:                                                                                                                                                                                                
            continue                                                                                                                                                                                             
        layer = int(m.group(1))                                                                                                                                                                                  
        idx = m.group(2)                                                                                                                                                                                         
        if idx is not None:                                                                                                                                                                                      
            out[layer].add(int(idx))                                                                                                                                                                             
        elif hasattr(tensor, "ndim") and tensor.ndim >= 2:                                                                                                                                                       
            out[layer].update(range(tensor.shape[0]))                                                                                                                                                            
    return out                                                                                                                                                                                                   
                                                                                                                                                                                                                
                                                                                                                                                                                                                
full_layers  = experts_per_layer(full_model.state_dict())                                                                                                                                                        
eval_layers  = experts_per_layer(eval_model.state_dict())   # the partial model                                                                                                                                  
miner_layers = experts_per_layer(miner_sd)                                                                                                                                                                       
                                                                                                                                                                                                                
all_layers = sorted(set(full_layers) | set(eval_layers) | set(miner_layers))

print(f"{'layer':>5}  {'full':>5}  {'eval-experts':<32}  {'miner-experts':<32}  {'hit':<24}  diff")     
print("-" * 150)

total_hit = 0                                      
total_missing = 0                                                                                       
total_extra = 0                                    
for L in all_layers:
    full_set  = full_layers.get(L, set())
    eval_set  = eval_layers.get(L, set())
    miner_set = miner_layers.get(L, set())                                                              

    hit                = sorted(eval_set & miner_set)  # actually overwritten on load                   
    missing_from_miner = sorted(eval_set - miner_set)  # eval expects, miner didn't ship
    extra_in_miner     = sorted(miner_set - eval_set)  # miner shipped, eval can't use
    total_hit     += len(hit)                                                                           
    total_missing += len(missing_from_miner)
    total_extra   += len(extra_in_miner)                                                                

    diff_parts = []                                                                                     
    if missing_from_miner:                         
        diff_parts.append(f"missing-from-miner={missing_from_miner}")
    if extra_in_miner:                                                                                  
        diff_parts.append(f"extra-in-miner={extra_in_miner}")
    diff_str = " ".join(diff_parts) if diff_parts else "ok"

    eval_str  = ",".join(map(str, sorted(eval_set)))  or "-"
    miner_str = ",".join(map(str, sorted(miner_set))) or "-"
    hit_str   = ",".join(map(str, hit))               or "-"                                            

    print(f"{L:>5}  {len(full_set):>5}  {eval_str:<32}  {miner_str:<32}  {hit_str:<24}  {diff_str}")    

print("-" * 150)                                                                                        
total_eval  = sum(len(s) for s in eval_layers.values())
total_miner = sum(len(s) for s in miner_layers.values())                                                
total_full  = sum(len(s) for s in full_layers.values())
print(f"  totals: full={total_full}  eval={total_eval}  miner={total_miner}")                           
print(f"  hit  (eval slots actually overwritten by miner): {total_hit}/{total_eval} "
    f"({100 * total_hit / max(total_eval, 1):.1f}%)")
print(f"  missing-from-miner (eval slots not covered):     {total_missing}")                            
print(f"  extra-in-miner    (miner shipped, eval unused):  {total_extra}")




layer   full  eval-experts                      miner-experts                     hit                       diff
------------------------------------------------------------------------------------------------------------------------------------------------------
    1     64  4,15,17,22,25,39,44,46            4,15,17,22,25,39,44,46            4,15,17,22,25,39,44,46    ok
    2     64  0,19,22,28,35,48,50,58            0,19,22,28,35,48,50,58            0,19,22,28,35,48,50,58    ok
    3     64  18,20,27,33,40,45,61,62           18,20,27,33,40,45,61,62           18,20,27,33,40,45,61,62   ok
    4     64  2,10,22,40,42,50,54,59            2,10,22,40,42,50,54,59            2,10,22,40,42,50,54,59    ok
    5     64  7,14,15,25,40,41,59,63            7,14,15,25,40,41,59,63            7,14,15,25,40,41,59,63    ok
    6     64  6,7,19,25,36,37,51,55             6,7,19,25,36,37,51,55             6,7,19,25,36,37,51,55     ok
    7     64  15,16,22,28,38,50,54,61           15,16,22,28,38,50,54,6

In [26]:
expert_manager.expert_group_assignment[1]

KeyError: 1